<a href="https://colab.research.google.com/github/edmundzen/dengue-early-warning/blob/main/member4_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai python-telegram-bot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 769.4/769.4 kB 14.3 MB/s eta 0:00:00


**MEMBER 4 : Gemini AI & Integration**

Responsibilities

• Retrieve prioritized dengue alerts from the BigQuery alert_queue table.
• Integrate AI-generated alert content (Recommendations, Executive Summary, and Alert Messages) from the alert_messages table.
• Merge alert data into a unified dataset for processing.
• Develop the Ask Sentinel chatbot to answer officer queries using the Gemini API.
•Implement the Human-in-the-Loop approval workflow, allowing officers to approve or reject alerts before action.
• Store officer decisions in the alert_decisions table for tracking and auditing
• Send approved dengue alerts to officers via Telegram.
• Export processed outputs and chat history as CSV files for reporting and dashboard integration.




## Workflow
Historical Dengue Cases + Weather Data
                 │
                 ▼
      AI Prediction & Risk Analysis
                 │
                 ▼
         Risk Ranking Engine
                 │
                 ▼
     BigQuery: alert_queue
 (Prioritized Dengue Alerts)
                 │
                 ▼
  BigQuery: alert_messages
(AI Recommendations, Executive Summary,
        Severity Alert Messages)
                 │
                 ▼
      Member 4 - Sentinel AI
                 │
        Merge Alert Data
                 │
        Ask Sentinel Chatbot
      (Powered by Gemini API)
                 │
 Human-in-the-Loop Approval
 (Approve / Reject by Officer)
                 │
                 ├──────────────► Store decisions in
                 │               alert_decisions
                 │
                 ├──────────────► Telegram Notification
                 │
                 ├──────────────► CSV Export
                 │
                 ▼
 Dashboard / Field Inspection Team

## Step 1: Import Required Libraries

In [2]:
import pandas as pd
import requests
import time

from google import genai
from google.cloud import bigquery
from google.colab import auth, userdata
from datetime import datetime

## Step 2: Authenticate with Google Cloud

In [3]:
auth.authenticate_user()

import subprocess
acct = subprocess.run(["gcloud","config","get-value","account"],
                      capture_output=True, text=True).stdout.strip()
print("Signed in as:", acct)

Signed in as: shraddhapal1505@gmail.com


## Step 3: Connect to BigQuery

In [4]:
PROJECT_ID = "dengue-early-warning"
bq_client = bigquery.Client(project=PROJECT_ID)
print("Connected!")

Connected!


##Step 4: Load alert queue
Purpose:
Load the prioritized dengue alerts generated by Member 2 from the BigQuery `alert_queue` table. These alerts will be processed by Sentinel AI to generate recommendations, executive summaries, severity alerts, and chatbot responses.

In [55]:
alerts = bq_client.query("""
SELECT *
FROM `dengue-early-warning.dengue_ew.alert_messages`
ORDER BY rank
""").to_dataframe()

alerts.head()

,rank,cell_id,risk_level,risk_score,zone,alert_message,executive_summary,recommendation
0,1,1.33425_103.8825,Critical,38.61,Tai Seng,Severity: CRITICAL\nReason: Highest 14-day cas...,Cell 1.33425_103.8825 is the top-ranked dengue...,1) Dispatch officers within 24h for door-to-do...
1,2,1.34775_103.95225,Critical,34.49,Changi Business Park,Severity: CRITICAL\nReason: Second-highest clu...,Cell 1.34775_103.95225 carries a critical scor...,1) Schedule inspection within 48h. 2) Clear st...
2,3,1.314_103.9275,High,24.20,Bedok,Severity: HIGH\nReason: Elevated case density ...,Cell 1.314_103.9275 is a high-risk emerging ho...,1) Add to this week inspection route. 2) Remov...
3,4,1.314_103.85325,High,23.00,Rochor,Severity: HIGH\nReason: 48 cases with a 2.3 fo...,Cell 1.314_103.85325 scores 23.0 and stands ou...,1) Inspect and trace the recurring source. 2) ...
4,5,1.3815_103.9365,High,21.72,Pasir Ris,"Severity: HIGH\nReason: 45 cases, forecast 2.1...",Cell 1.3815_103.9365 (score 21.7) shows steady...,1) Include in weekly inspection. 2) Clear stag...


In [8]:
print("Columns in alert_queue:")

print(inspection_df.columns.tolist())

Columns in alert_queue:
['alert_id', 'cell_id', 'lat', 'lon', 'risk_score', 'risk_level', 'forecast_value', 'case_density_14d', 'recurrence', 'rank', 'created_at']


In [9]:
inspection_df[[
    "cell_id",
    "risk_level",
    "risk_score"
]].head()

,cell_id,risk_level,risk_score
0,1.33425_103.8825,Critical,38.61
1,1.34775_103.95225,Critical,34.49
2,1.314_103.9275,High,24.20
3,1.314_103.85325,High,23.00
4,1.3815_103.9365,High,21.72


In [57]:
PROJECT_ID = "dengue-early-warning"

bq_client = bigquery.Client(project=PROJECT_ID)

In [58]:
inspection_df = bq_client.query(query).to_dataframe()

In [59]:
inspection_df.head()

,alert_id,cell_id,lat,lon,risk_score,risk_level,forecast_value,case_density_14d,recurrence,rank,created_at
0,fe2710b8-0122-4528-9179-0f395c16c0d3,1.33425_103.8825,1.33425,103.88250,38.61,Critical,4.081365,86.0,0.100819,1,2026-07-23 13:36:56.166241+00:00
1,85dbe860-a710-4a34-a386-11191f50cf1d,1.34775_103.95225,1.34775,103.95225,34.49,Critical,3.601444,76.0,0.063460,2,2026-07-23 13:36:56.166241+00:00
2,8a93a1da-b59f-4f63-bdfa-be371f658b55,1.314_103.9275,1.31400,103.92750,24.20,High,2.419172,51.0,0.075230,3,2026-07-23 13:36:56.166241+00:00
3,ae78f537-b055-4250-931a-9bf1e7004ad0,1.314_103.85325,1.31400,103.85325,23.00,High,2.275486,48.0,0.315763,4,2026-07-23 13:36:56.166241+00:00
4,37834b62-90bb-4bf1-ad18-2ae087c1ccf1,1.3815_103.9365,1.38150,103.93650,21.72,High,2.133953,45.0,0.026612,5,2026-07-23 13:36:56.166241+00:00


In [13]:
print(inspection_df["risk_score"].tolist())

[38.61, 34.49, 24.2, 23.0, 21.72, 21.31, 20.07]


## Step 5: Configure Gemini API

In [15]:
from google import genai
from google.colab import userdata

# Read API Key from Colab Secrets
api_key = userdata.get("GEMINI_KEY")

# Create Gemini Client
gemini_client = genai.Client(api_key=api_key)

print("✅ Gemini Connected Successfully")

✅ Gemini Connected Successfully


## Step 6: Generate AI Recommendation

In [16]:
def generate_recommendation(row):

    prompt = f"""
You are a Public Health Officer.

Analyze this dengue alert.

Cell ID: {row['cell_id']}
Risk Score: {row['risk_score']}
Risk Level: {row['risk_level']}

Provide your answer in exactly this format:

Risk Summary:
(2 short lines)

Inspection Priority:
(One sentence)

Recommended Actions:
• Action 1
• Action 2
• Action 3

Expected Impact:
(One short sentence)

Keep the response under 150 words.
"""

    try:

        response = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:

        return f"Error: {e}"

In [18]:
inspection_df[[
    "cell_id",
    "risk_level",
    "risk_score",
    "AI_Recommendation"
]].head()

,cell_id,risk_level,risk_score,AI_Recommendation
0,1.33425_103.8825,Critical,38.61,Risk Summary:\nCell 1.33425_103.8825 exhibits ...
1,1.34775_103.95225,Critical,34.49,Risk Summary:\nCell 1.34775_103.95225 shows a ...
2,1.314_103.9275,High,24.20,Risk Summary:\nHigh dengue transmission risk f...
3,1.314_103.85325,High,23.00,Risk Summary:\nCell 1.314_103.85325 shows a hi...
4,1.3815_103.9365,High,21.72,Risk Summary:\nCell 1.3815_103.9365 exhibits a...


## Step 7: Generate Executive Summary

In [26]:
import time

def executive_summary(row):

    prompt = f"""
You are a dengue surveillance analyst.

Summarize this alert in less than 70 words.

Cell ID: {row['cell_id']}
Risk Level: {row['risk_level']}
Risk Score: {row['risk_score']}
Forecast Value: {row['forecast_value']}
Case Density: {row['case_density_14d']}
Recurrence: {row['recurrence']}
"""

    for attempt in range(3):

        try:

            response = gemini_client.models.generate_content(
                model="gemini-3.6-flash",
                contents=prompt
            )

            return response.text

        except Exception:

            time.sleep(5)

    return "Executive Summary unavailable."
    return "Executive summary unavailable."

In [27]:
print(inspection_df.columns.tolist())

['alert_id', 'cell_id', 'lat', 'lon', 'risk_score', 'risk_level', 'forecast_value', 'case_density_14d', 'recurrence', 'rank', 'created_at', 'AI_Recommendation']


In [29]:
inspection_df[[
    "cell_id",
    "Executive_Summary"
]].head()

,cell_id,Executive_Summary
0,1.33425_103.8825,**Critical Dengue Alert Summary**\n\nCell ID *...
1,1.34775_103.95225,**CRITICAL DENGUE ALERT** (Cell ID: 1.34775_10...
2,1.314_103.9275,**Dengue Surveillance Alert**\n\nLocation Cell...
3,1.314_103.85325,**Dengue Surveillance Alert Summary**\n\n* **L...
4,1.3815_103.9365,**High-Risk Dengue Alert:** \n\nLocation Cell ...


## Step 8: Generate Severity Alert Messages

Purpose:
Generate severity-based alert messages for officers.

In [61]:
def generate_alert(row):

    prompt = f"""
You are the Officer of the Day.

Generate a professional dengue alert.

Cell ID: {row['cell_id']}
Risk Level: {row['risk_level']}
Risk Score: {row['risk_score']}

Write:

Alert Level:
Reason:
Immediate Action:

Keep it under 80 words.
"""

    try:

        response = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:

        return f"Error: {e}"

In [66]:
inspection_df = inspection_df.merge(
    alerts[
        [
            "cell_id",
            "alert_message",
            "executive_summary",
            "recommendation"
        ]
    ],
    on="cell_id",
    how="left"
)

In [67]:
#rename
inspection_df.rename(columns={
    "executive_summary": "Executive_Summary",
    "recommendation": "AI_Recommendation"
}, inplace=True)

In [68]:
inspection_df[["cell_id","risk_level","alert_message"]].head()

,cell_id,risk_level,alert_message
0,1.33425_103.8825,Critical,Severity: CRITICAL\nReason: Highest 14-day cas...
1,1.34775_103.95225,Critical,Severity: CRITICAL\nReason: Second-highest clu...
2,1.314_103.9275,High,Severity: HIGH\nReason: Elevated case density ...
3,1.314_103.85325,High,Severity: HIGH\nReason: 48 cases with a 2.3 fo...
4,1.3815_103.9365,High,"Severity: HIGH\nReason: 45 cases, forecast 2.1..."


## Step 9: Human-in-the-Loop Approval

Purpose:
Every AI-generated alert is reviewed by the Officer of the Day before execution.
Only approved alerts proceed to notification.

In [33]:
from datetime import datetime

def officer_decision(row, decision, officer, reason):

    return {
        "alert_id": row["alert_id"],
        "cell_id": row["cell_id"],
        "decision": decision,
        "officer": officer,
        "reason": reason,
        "decided_at": datetime.now()
    }


## Step 10: Officer Approval Workflow

In [69]:
decision_list = []

for _, row in inspection_df.iterrows():

    if row["risk_score"] >= 30:

        decision = "Approved"
        reason = "Critical dengue risk"

    elif row["risk_score"] >= 20:

        decision = "Approved"
        reason = "High dengue risk"

    else:

        decision = "Rejected"
        reason = "Risk below threshold"

    decision_list.append(

        officer_decision(
            row,
            decision,
            "Health Officer",
            reason
        )
    )

decision_df = pd.DataFrame(decision_list)

decision_df.head()

,alert_id,cell_id,decision,officer,reason,decided_at
0,fe2710b8-0122-4528-9179-0f395c16c0d3,1.33425_103.8825,Approved,Health Officer,Critical dengue risk,2026-07-24 07:40:53.480982
1,85dbe860-a710-4a34-a386-11191f50cf1d,1.34775_103.95225,Approved,Health Officer,Critical dengue risk,2026-07-24 07:40:53.481072
2,8a93a1da-b59f-4f63-bdfa-be371f658b55,1.314_103.9275,Approved,Health Officer,High dengue risk,2026-07-24 07:40:53.481135
3,ae78f537-b055-4250-931a-9bf1e7004ad0,1.314_103.85325,Approved,Health Officer,High dengue risk,2026-07-24 07:40:53.481192
4,37834b62-90bb-4bf1-ad18-2ae087c1ccf1,1.3815_103.9365,Approved,Health Officer,High dengue risk,2026-07-24 07:40:53.481266


## Step 11: Ask Sentinel Chatbot

Purpose:
Allow officers to ask questions about a dengue alert and receive grounded AI explanations.

In [70]:
def ask_sentinel(row, question):

    prompt = f"""
You are Sentinel AI.

You assist dengue surveillance officers.

Current Alert

Alert ID: {row['alert_id']}
Cell ID: {row['cell_id']}
Risk Level: {row['risk_level']}
Risk Score: {row['risk_score']}
Forecast Value: {row['forecast_value']}
Case Density: {row['case_density_14d']}
Recurrence: {row['recurrence']}

Officer Question:
{question}

Answer using this format:

Risk Summary:
Inspection Priority:
Recommended Actions:
Expected Impact:

Keep the answer under 150 words.
"""

    try:

        response = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=prompt
        )

        return response.text

    except Exception as e:

        return f"Error: {e}"

In [36]:
#test chatbot
question = "Why is this area high risk?"

answer = ask_sentinel(
    inspection_df.iloc[0],
    question
)

print(answer)

**Risk Summary:**
This cell is at Critical risk (Score: 38.61) due to an exceptionally high case density of 86.0 and an elevated forecast value of 4.08, signaling intense active dengue transmission in the area.

**Inspection Priority:**
Immediate (Priority 1) — Requires field deployment within 24 hours.

**Recommended Actions:**
* Conduct intensive search-and-destroy inspections for mosquito breeding sources.
* Deploy thermal fogging or space spraying to target adult vector populations.
* Issue localized community alerts and engage residents on preventive measures.

**Expected Impact:**
Rapid vector control will disrupt transmission chains, reducing vector density and preventing further secondary cases in this outbreak hotspot.


In [37]:
#multiple questions
questions = [

    "Why is this area high risk?",

    "What actions should officers take?",

    "Should this alert be approved?"
]

for q in questions:

    print("="*60)

    print("Question:", q)

    print()

    print(ask_sentinel(
        inspection_df.iloc[0],
        q
    ))

Question: Why is this area high risk?

**Risk Summary:**  
This area is at Critical risk (Score: 38.61) driven by an exceptionally high case density (86.0) and a high forecasted case increase (4.08), indicating intense active transmission.

**Inspection Priority:**  
Immediate (Tier 1). Priority focus on outdoor drains, construction sites, and residential common areas within Cell 1.33425_103.8825.

**Recommended Actions:**  
1. Deploy vector control teams for intensive breeding habitat search-and-destroy operations.  
2. Execute thermal fogging or space spraying to suppress adult mosquito populations.  
3. Issue targeted dengue alerts to residents and premise managers.

**Expected Impact:**  
Rapid reduction of vector density, disruption of local transmission chains, and prevention of further cluster expansion.
Question: What actions should officers take?

**Risk Summary:**  
Cell 1.33425_103.8825 is at a **Critical** risk level (Score: 38.61) with a high case density of 86.0, indicati

## Step 12: Telegram Notification

Purpose:
Send approved dengue alerts to field officers through Telegram.

In [47]:
import requests

BOT_TOKEN = userdata.get("BOT_TOKEN")
CHAT_ID = userdata.get("CHAT_ID")

In [48]:
def send_telegram_alert(row):

    message = f"""
🚨 DENGUE EARLY WARNING

Cell ID: {row['cell_id']}

Risk Level: {row['risk_level']}

Risk Score: {row['risk_score']}

Executive Summary

{row['Executive_Summary']}

AI Recommendation

{row['AI_Recommendation']}
"""

    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"

    payload = {

        "chat_id": CHAT_ID,

        "text": message
    }

    requests.post(
        url,
        json=payload
    )

In [49]:
#send approval message
for _, row in inspection_df.iterrows():

    if row["risk_score"] >= 20:

        send_telegram_alert(row)

print("✅ Telegram alerts sent.")

✅ Telegram alerts sent.


## Step 13: Export Results to CSV

Purpose:
Save AI-generated outputs and officer decisions for reporting and dashboard integration.

In [50]:
# Save AI recommendations, summaries, and alerts
inspection_df.to_csv("AI_Recommendation.csv", index=False)

# Save officer decisions
decision_df.to_csv("alert_decisions.csv", index=False)

print("✅ CSV files exported successfully.")

✅ CSV files exported successfully.


## Step 14: Save Sentinel Chat History

Purpose:
Store officer questions and Sentinel AI responses for auditing and future analysis.

In [51]:
chat_history = []

questions = [
    "Why is this area high risk?",
    "What actions should officers take?",
    "Should this alert be approved?"
]

for q in questions:

    ans = ask_sentinel(
        inspection_df.iloc[0],
        q
    )

    chat_history.append({

        "alert_id": inspection_df.iloc[0]["alert_id"],

        "question": q,

        "answer": ans

    })

chat_df = pd.DataFrame(chat_history)

chat_df.to_csv("sentinel_chat_history.csv", index=False)

print("✅ Chat history saved.")

✅ Chat history saved.


## Step 15: Upload Officer Decisions to BigQuery

Purpose:
Store officer approval decisions in BigQuery for dashboard visualization and tracking.

In [52]:
job = bq_client.load_table_from_dataframe(

    decision_df,

    "dengue-early-warning.dengue_ew.alert_decisions",

    job_config=bigquery.LoadJobConfig(

        write_disposition="WRITE_TRUNCATE"

    )

)

job.result()

print("✅ alert_decisions uploaded to BigQuery.")

✅ alert_decisions uploaded to BigQuery.


## Step 16: Final Outputs

Purpose:
Preview the generated outputs before deployment.

In [44]:
print("Inspection Data")

inspection_df.head()

Inspection Data


,alert_id,cell_id,lat,lon,risk_score,risk_level,forecast_value,case_density_14d,recurrence,rank,created_at,AI_Recommendation,Executive_Summary,alert_message
0,fe2710b8-0122-4528-9179-0f395c16c0d3,1.33425_103.8825,1.33425,103.88250,38.61,Critical,4.081365,86.0,0.100819,1,2026-07-23 13:36:56.166241+00:00,Risk Summary:\nCell 1.33425_103.8825 exhibits ...,**Critical Dengue Alert Summary**\n\nCell ID *...,**Alert Level:** CRITICAL (Risk Score: 38.61) ...
1,85dbe860-a710-4a34-a386-11191f50cf1d,1.34775_103.95225,1.34775,103.95225,34.49,Critical,3.601444,76.0,0.063460,2,2026-07-23 13:36:56.166241+00:00,Risk Summary:\nCell 1.34775_103.95225 shows a ...,**CRITICAL DENGUE ALERT** (Cell ID: 1.34775_10...,**Alert Level:** CRITICAL (Risk Score: 34.49) ...
2,8a93a1da-b59f-4f63-bdfa-be371f658b55,1.314_103.9275,1.31400,103.92750,24.20,High,2.419172,51.0,0.075230,3,2026-07-23 13:36:56.166241+00:00,Risk Summary:\nHigh dengue transmission risk f...,**Dengue Surveillance Alert**\n\nLocation Cell...,**Alert Level:** HIGH \n\n**Reason:** Cell ID...
3,ae78f537-b055-4250-931a-9bf1e7004ad0,1.314_103.85325,1.31400,103.85325,23.00,High,2.275486,48.0,0.315763,4,2026-07-23 13:36:56.166241+00:00,Risk Summary:\nCell 1.314_103.85325 shows a hi...,**Dengue Surveillance Alert Summary**\n\n* **L...,**Alert Level:** HIGH (Risk Score: 23.0)\n\n**...
4,37834b62-90bb-4bf1-ad18-2ae087c1ccf1,1.3815_103.9365,1.38150,103.93650,21.72,High,2.133953,45.0,0.026612,5,2026-07-23 13:36:56.166241+00:00,Risk Summary:\nCell 1.3815_103.9365 exhibits a...,**High-Risk Dengue Alert:** \n\nLocation Cell ...,**Alert Level:** HIGH (Risk Score: 21.72)\n\n*...


In [53]:
print("Officer Decisions")

decision_df.head()

Officer Decisions


,alert_id,cell_id,decision,officer,reason,decided_at
0,fe2710b8-0122-4528-9179-0f395c16c0d3,1.33425_103.8825,Approved,Health Officer,Critical dengue risk,2026-07-24 07:09:20.216458
1,85dbe860-a710-4a34-a386-11191f50cf1d,1.34775_103.95225,Approved,Health Officer,Critical dengue risk,2026-07-24 07:09:20.216551
2,8a93a1da-b59f-4f63-bdfa-be371f658b55,1.314_103.9275,Approved,Health Officer,High dengue risk,2026-07-24 07:09:20.216620
3,ae78f537-b055-4250-931a-9bf1e7004ad0,1.314_103.85325,Approved,Health Officer,High dengue risk,2026-07-24 07:09:20.216679
4,37834b62-90bb-4bf1-ad18-2ae087c1ccf1,1.3815_103.9365,Approved,Health Officer,High dengue risk,2026-07-24 07:09:20.216734


In [54]:
print("Sentinel Chat History")

chat_df.head()

Sentinel Chat History


,alert_id,question,answer
0,fe2710b8-0122-4528-9179-0f395c16c0d3,Why is this area high risk?,**Risk Summary:**\nThis area is at Critical ri...
1,fe2710b8-0122-4528-9179-0f395c16c0d3,What actions should officers take?,Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...
2,fe2710b8-0122-4528-9179-0f395c16c0d3,Should this alert be approved?,Error: 429 RESOURCE_EXHAUSTED. {'error': {'cod...


##Conclusion

Sentinel AI successfully completed the following workflow:

1. Retrieved prioritized dengue alerts from BigQuery.
2. Generated AI Recommendations using Gemini.
3. Created Executive Summaries for health officers.
4. Generated Severity Alert Messages.
5. Supported Grounded Q&A through the Ask Sentinel chatbot.
6. Enabled Human-in-the-Loop approval decisions.
7. Sent approved alerts via Telegram.
8. Exported reports to CSV.
9. Uploaded officer decisions to BigQuery.

The system provides an end-to-end AI-assisted dengue early warning workflow for public health surveillance.